> **Work In Progress** -- This notebook is a preliminary version. Content may change before the session. Check back for updates.

# Session 1 -- ML Landscape & EDA

**B4990 - Artificial Intelligence** | Universidad Nacional de Rio Negro  
**Date:** Monday, March 2, 2026 | **Instructor:** Federico Tula Rovaletti

---

**Objectives:**
1. Understand what ML is and distinguish supervised, unsupervised, and reinforcement learning.
2. Perform exploratory data analysis (EDA) with pandas and matplotlib -- both visual and statistical.
3. Use NumPy vectorized operations and understand why they matter.
4. Develop intuition for the bias/variance tradeoff, then see the math and the empirical proof.
5. Understand why aggregated statistics can be misleading (Simpson's Paradox, spurious correlations, Anscombe's quartet).

**Tools:** Python 3.14, NumPy, pandas, matplotlib, scipy.stats (no seaborn, no sklearn yet).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Reproducibility
np.random.seed(42)

# Plot settings
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

---
## Part 1: What is Machine Learning?

### Traditional Programming vs Machine Learning

**Traditional programming:** You write explicit rules.  
- Input: data + rules --> Output: answers

**Machine learning:** The computer learns rules from data.  
- Input: data + answers --> Output: rules (a model)

### The Three Types of Learning

| Type | What you have | Goal | Example |
|------|--------------|------|--------|
| **Supervised** | Labeled data (input + correct answer) | Predict the answer for new inputs | Spam filter, house price prediction |
| **Unsupervised** | Unlabeled data (input only) | Find hidden structure | Customer segmentation, anomaly detection |
| **Reinforcement** | Environment + rewards | Learn a strategy to maximize reward | Game-playing AI, robot navigation |

### "Is This ML?" Exercise

For each scenario below, decide: **(a) Is it ML? (b) If yes, what type?**

1. A program that sorts a list of numbers from smallest to largest.
2. An email system that learns to separate spam from legitimate messages based on past examples.
3. A system that groups customers into segments based on their purchasing behavior, without predefined categories.
4. A calculator app that computes square roots.
5. A chess engine that improves by playing millions of games against itself.
6. A program that predicts tomorrow's temperature based on historical weather data.

**Answers:**

1. **Not ML.** Sorting is a deterministic algorithm -- no learning from data.
2. **ML -- Supervised.** Labeled examples (spam/not spam) are used to learn a classification rule.
3. **ML -- Unsupervised.** No labels; the algorithm discovers groups on its own.
4. **Not ML.** A fixed mathematical formula -- no learning involved.
5. **ML -- Reinforcement Learning.** The agent (chess engine) learns a policy by receiving rewards (win/lose).
6. **ML -- Supervised.** Historical data with input features (past weather) and labels (actual temperature).

---
### Bias/Variance Tradeoff -- Intuition

One of the most important ideas in ML: **your model can fail in two opposite ways.**

- **Bias (underfitting):** The model is too simple. It cannot capture the real pattern.
- **Variance (overfitting):** The model is too complex. It memorizes the training data, including the noise.

Let's see this visually.

In [ ]:
# Generate noisy sine wave data
np.random.seed(42)
n_points = 30
x = np.linspace(0, 2 * np.pi, n_points)
y_true = np.sin(x)
noise = np.random.normal(0, 0.3, n_points)
y = y_true + noise

# Smooth x for plotting fitted curves
x_smooth = np.linspace(0, 2 * np.pi, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

titles = ['Underfitting (degree 1)', 'Good fit (degree 4)', 'Overfitting (degree 15)']
degrees = [1, 4, 15]

for ax, degree, title in zip(axes, degrees, titles):
    # Fit polynomial
    coeffs = np.polyfit(x, y, degree)
    poly = np.poly1d(coeffs)
    
    # Plot
    ax.scatter(x, y, color='steelblue', s=30, alpha=0.7, label='Training data')
    ax.plot(x_smooth, np.sin(x_smooth), 'g--', alpha=0.5, label='True function')
    ax.plot(x_smooth, poly(x_smooth), 'r-', linewidth=2, label=f'Polynomial (deg {degree})')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.savefig('bias_variance.png', dpi=100, bbox_inches='tight')
plt.show()

print("Left:   HIGH bias, LOW variance   -- the line is too simple to capture the curve.")
print("Center: LOW bias, LOW variance     -- captures the pattern without fitting the noise.")
print("Right:  LOW bias, HIGH variance    -- fits every point, including noise. Will fail on new data.")

**Key takeaway:** The goal of ML is to find the model complexity sweet spot -- complex enough to capture the real pattern, simple enough to generalize to new data.

We will return to this idea in almost every session.

---
### Bias-Variance Decomposition -- The Math

The visual intuition above is nice, but let's prove it. This is one of the most important derivations in statistical learning theory.

**Setup:** We have a true function $f(x)$ and we observe $y = f(x) + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2)$.

We train a model $\hat{f}(x)$ on a dataset $\mathcal{D}$ sampled from this distribution. Different datasets give different $\hat{f}$.

**Goal:** Decompose the expected prediction error at a point $x$.

$$
\text{EPE}(x) = E_{\mathcal{D}, \epsilon}\left[(y - \hat{f}(x))^2\right]
$$

**Step 1:** Expand the square by adding and subtracting $f(x)$.

$$
E\left[(y - \hat{f}(x))^2\right] = E\left[((y - f(x)) + (f(x) - \hat{f}(x)))^2\right]
$$

$$
= E\left[(y - f(x))^2\right] + E\left[(f(x) - \hat{f}(x))^2\right] + 2\,E\left[(y - f(x))(f(x) - \hat{f}(x))\right]
$$

**Step 2:** The cross-term vanishes.

Since $y - f(x) = \epsilon$ is independent of $\hat{f}(x)$ (noise is independent of which dataset we sampled to train), and $E[\epsilon] = 0$:

$$
E\left[\epsilon \cdot (f(x) - \hat{f}(x))\right] = E[\epsilon] \cdot E[f(x) - \hat{f}(x)] = 0
$$

**Step 3:** First term is just the noise variance.

$$
E\left[(y - f(x))^2\right] = E[\epsilon^2] = \sigma^2
$$

**Step 4:** Decompose the second term by adding and subtracting $E_{\mathcal{D}}[\hat{f}(x)]$.

$$
E\left[(f(x) - \hat{f}(x))^2\right] = E\left[((f(x) - E[\hat{f}(x)]) + (E[\hat{f}(x)] - \hat{f}(x)))^2\right]
$$

$$
= (f(x) - E[\hat{f}(x)])^2 + E\left[(\hat{f}(x) - E[\hat{f}(x)])^2\right] + 2(f(x) - E[\hat{f}(x)]) \cdot \underbrace{E[\hat{f}(x) - E[\hat{f}(x)]]}_{=0}
$$

$$
= \underbrace{(f(x) - E[\hat{f}(x)])^2}_{\text{Bias}^2} + \underbrace{E\left[(\hat{f}(x) - E[\hat{f}(x)])^2\right]}_{\text{Variance}}
$$

**Final result:**

$$
\boxed{E\left[(y - \hat{f}(x))^2\right] = \text{Bias}[\hat{f}(x)]^2 + \text{Var}[\hat{f}(x)] + \sigma^2}
$$

- **Bias$^2$:** How far off is the average prediction from the truth? (systematic error from model assumptions)
- **Variance:** How much does the prediction change across different training sets? (sensitivity to data)
- **$\sigma^2$ (irreducible error):** Noise in the data. No model can do better than this.

**The tradeoff:** Increasing model complexity typically decreases bias but increases variance, and vice versa. The optimal model minimizes the sum.

### Empirical Bias-Variance Decomposition

Let's verify the decomposition by simulation. We will:
1. Define a true function $f(x) = \sin(x)$ with noise $\sigma = 0.3$.
2. Generate **many** independent training datasets from this distribution.
3. For each dataset, fit polynomial models of various degrees.
4. At a grid of test points, compute the actual bias$^2$, variance, and expected error.
5. Check that $\text{Error} \approx \text{Bias}^2 + \text{Variance} + \sigma^2$.

In [ ]:
np.random.seed(42)

# True function and noise
f_true = lambda x: np.sin(x)
sigma = 0.3
n_train = 25          # points per training set
n_datasets = 200      # number of independent training sets
degrees = [1, 2, 3, 4, 5, 7, 9, 12, 15]

# Fixed test points where we evaluate
x_test = np.linspace(0.1, 2 * np.pi - 0.1, 50)
f_test = f_true(x_test)

# For each degree, collect predictions across all datasets
results = {}

for deg in degrees:
    # Matrix: (n_datasets, n_test_points)
    preds = np.zeros((n_datasets, len(x_test)))
    
    for i in range(n_datasets):
        # Generate a new training set
        x_train = np.sort(np.random.uniform(0, 2 * np.pi, n_train))
        y_train = f_true(x_train) + np.random.normal(0, sigma, n_train)
        
        # Fit polynomial
        coeffs = np.polyfit(x_train, y_train, deg)
        poly = np.poly1d(coeffs)
        preds[i, :] = poly(x_test)
    
    # Compute bias^2 and variance at each test point, then average
    mean_pred = preds.mean(axis=0)                     # E[f_hat(x)]
    bias_sq = ((f_test - mean_pred) ** 2).mean()       # avg Bias^2 over x
    variance = preds.var(axis=0).mean()                # avg Var over x
    
    # Expected MSE: average over datasets and test points
    # For each dataset, MSE vs noisy targets
    mse_list = []
    for i in range(n_datasets):
        y_test_noisy = f_test + np.random.normal(0, sigma, len(x_test))
        mse_list.append(((y_test_noisy - preds[i, :]) ** 2).mean())
    expected_mse = np.mean(mse_list)
    
    results[deg] = {
        'bias_sq': bias_sq,
        'variance': variance,
        'noise': sigma ** 2,
        'expected_mse': expected_mse,
        'decomposition_sum': bias_sq + variance + sigma ** 2,
    }

# Print table
print(f"{'Degree':>6}  {'Bias^2':>8}  {'Variance':>8}  {'Noise':>6}  {'B^2+V+N':>8}  {'Actual MSE':>10}")
print("-" * 60)
for deg in degrees:
    r = results[deg]
    print(f"{deg:>6d}  {r['bias_sq']:>8.4f}  {r['variance']:>8.4f}  {r['noise']:>6.4f}  "
          f"{r['decomposition_sum']:>8.4f}  {r['expected_mse']:>10.4f}")

In [ ]:
# Plot the bias-variance tradeoff curve
degs = list(results.keys())
bias_sq_vals = [results[d]['bias_sq'] for d in degs]
var_vals = [results[d]['variance'] for d in degs]
noise_vals = [results[d]['noise'] for d in degs]
mse_vals = [results[d]['expected_mse'] for d in degs]
decomp_vals = [results[d]['decomposition_sum'] for d in degs]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(degs, bias_sq_vals, 'b-o', linewidth=2, markersize=6, label='Bias$^2$')
ax.plot(degs, var_vals, 'r-s', linewidth=2, markersize=6, label='Variance')
ax.axhline(y=sigma**2, color='gray', linestyle='--', linewidth=1.5, label=f'Irreducible noise ($\\sigma^2={sigma**2}$)')
ax.plot(degs, decomp_vals, 'k-^', linewidth=2.5, markersize=7, label='Bias$^2$ + Variance + $\\sigma^2$')
ax.plot(degs, mse_vals, 'g--d', linewidth=2, markersize=6, alpha=0.8, label='Actual Expected MSE')

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13)
ax.set_ylabel('Error', fontsize=13)
ax.set_title('Bias-Variance Tradeoff (Empirical)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xticks(degs)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bias_variance_decomposition.png', dpi=100, bbox_inches='tight')
plt.show()

print("Observations:")
print("- Bias^2 decreases as model complexity increases (model can represent more shapes).")
print("- Variance increases with complexity (model becomes more sensitive to the specific training set).")
print("- The green dashed line (actual MSE) closely tracks the black solid line (decomposition sum).")
print("- The minimum total error occurs at an intermediate complexity -- the sweet spot.")
print(f"- No model can go below sigma^2 = {sigma**2} (irreducible noise).")

In [ ]:
# Visualize what "variance" looks like: plot predictions from 20 different training sets
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
demo_degrees = [1, 4, 15]
demo_titles = ['Degree 1 (High Bias, Low Var)', 'Degree 4 (Sweet Spot)', 'Degree 15 (Low Bias, High Var)']
x_plot = np.linspace(0.1, 2 * np.pi - 0.1, 200)

for ax, deg, title in zip(axes, demo_degrees, demo_titles):
    for i in range(20):
        x_train = np.sort(np.random.uniform(0, 2 * np.pi, n_train))
        y_train = f_true(x_train) + np.random.normal(0, sigma, n_train)
        coeffs = np.polyfit(x_train, y_train, deg)
        poly = np.poly1d(coeffs)
        ax.plot(x_plot, poly(x_plot), 'steelblue', alpha=0.2, linewidth=1)
    
    ax.plot(x_plot, f_true(x_plot), 'r-', linewidth=2.5, label='True $f(x)$')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=9)
    ax.set_xlabel('x')

plt.suptitle('Each blue line = model trained on a different dataset (20 shown)', fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

print("Left: All 20 models look similar (low variance) but they all miss the curve (high bias).")
print("Center: Models cluster around the true function with moderate spread.")
print("Right: Models wildly disagree, especially at the edges (high variance), but on average they hit the truth (low bias).")

---
## Part 2: Exploratory Data Analysis (EDA)

**Rule #1 of ML:** Always look at your data before modeling.

We will use the Palmer Penguins dataset -- measurements of 344 penguins from three species in Antarctica. We create it inline so no downloads are needed.

In [ ]:
# ============================================================
# Palmer Penguins dataset (inline, no external dependencies)
# 344 penguins, 3 species, 3 islands
# Based on: Gorman KB, Williams TD, Fraser WR (2014)
# ============================================================

np.random.seed(42)

def _make_penguins():
    """Create a realistic penguins dataset inline."""
    records = []
    
    # Species parameters: (bill_length_mean, bill_length_std,
    #                      bill_depth_mean, bill_depth_std,
    #                      flipper_length_mean, flipper_length_std,
    #                      body_mass_mean, body_mass_std)
    species_params = {
        'Adelie': (38.8, 2.7, 18.3, 1.2, 190, 6.5, 3700, 460,
                   ['Torgersen', 'Biscoe', 'Dream'], [24, 44, 84]),
        'Chinstrap': (48.8, 3.3, 18.4, 1.1, 196, 7.1, 3730, 380,
                      ['Dream'], [68]),
        'Gentoo': (47.5, 3.1, 15.0, 1.0, 217, 6.5, 5080, 500,
                   ['Biscoe'], [124]),
    }
    
    for species, params in species_params.items():
        bl_m, bl_s, bd_m, bd_s, fl_m, fl_s, bm_m, bm_s, islands, counts = (
            params[0], params[1], params[2], params[3],
            params[4], params[5], params[6], params[7],
            params[8], params[9]
        )
        for island, n in zip(islands, counts):
            bill_length = np.random.normal(bl_m, bl_s, n).round(1)
            bill_depth = np.random.normal(bd_m, bd_s, n).round(1)
            flipper_length = np.random.normal(fl_m, fl_s, n).round(0).astype(int)
            body_mass = np.random.normal(bm_m, bm_s, n).round(-1).astype(int)
            sex = np.random.choice(['male', 'female'], n)
            
            for i in range(n):
                records.append({
                    'species': species,
                    'island': island,
                    'bill_length_mm': bill_length[i],
                    'bill_depth_mm': bill_depth[i],
                    'flipper_length_mm': flipper_length[i],
                    'body_mass_g': body_mass[i],
                    'sex': sex[i],
                })
    
    df = pd.DataFrame(records)
    
    # Introduce some missing values (realistic -- field measurements sometimes fail)
    rng = np.random.default_rng(42)
    for col in ['bill_length_mm', 'bill_depth_mm', 'sex']:
        mask = rng.random(len(df)) < 0.03  # ~3% missing
        df.loc[mask, col] = np.nan
    
    # Shuffle rows
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

df = _make_penguins()
print(f"Dataset created: {df.shape[0]} penguins, {df.shape[1]} columns")

### Step 1: First Look

In [ ]:
# Always start here: look at the first few rows
df.head(10)

In [ ]:
# Shape: how many rows and columns?
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

In [ ]:
# dtypes and non-null counts
df.info()

### Step 2: Missing Values

In [ ]:
# Count missing values per column
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0])
print(f"\nTotal missing: {df.isna().sum().sum()}")
print(f"Percentage missing: {df.isna().sum().sum() / df.size * 100:.2f}%")

### Step 3: Summary Statistics

In [ ]:
# Numeric summary
df.describe()

In [ ]:
# Categorical columns
print("Species distribution:")
print(df['species'].value_counts())
print(f"\nIsland distribution:")
print(df['island'].value_counts())
print(f"\nSex distribution:")
print(df['sex'].value_counts())

### Step 4: Visualizations

#### Histograms -- Distribution of Numeric Features

In [ ]:
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col].dropna(), bins=25, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Distribution of Numeric Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**What do you notice?**
- `bill_length_mm` looks bimodal -- probably two groups with different bill lengths.
- `flipper_length_mm` also looks bimodal.
- `body_mass_g` has a long right tail -- some penguins are much heavier.

### Step 5: Statistical EDA -- Beyond Eyeballing

Histograms give you a rough picture, but statistics let you be precise. Let's go beyond `.describe()` and use proper statistical tools.

#### Skewness and Kurtosis

- **Skewness** measures asymmetry. Zero = symmetric. Positive = right-tailed. Negative = left-tailed.
- **Kurtosis** measures tail heaviness relative to a normal distribution. Zero (Fisher convention) = normal-like tails. Positive = heavier tails. Negative = lighter tails.

Rule of thumb: skewness between -0.5 and 0.5 is approximately symmetric.

In [ ]:
# Compute skewness and kurtosis for all numeric columns
print(f"{'Column':<20s} {'Skewness':>10s} {'Kurtosis':>10s}  Interpretation")
print("-" * 70)

for col in numeric_cols:
    data = df[col].dropna()
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)  # Fisher convention (excess kurtosis, normal=0)
    
    # Interpret skewness
    if abs(skew) < 0.5:
        skew_interp = "~symmetric"
    elif skew > 0:
        skew_interp = "right-skewed"
    else:
        skew_interp = "left-skewed"
    
    # Interpret kurtosis
    if abs(kurt) < 0.5:
        kurt_interp = "normal-like tails"
    elif kurt > 0:
        kurt_interp = "heavy tails"
    else:
        kurt_interp = "light tails"
    
    print(f"{col:<20s} {skew:>10.3f} {kurt:>10.3f}  {skew_interp}, {kurt_interp}")

#### Shapiro-Wilk Test for Normality

The Shapiro-Wilk test is one of the most powerful tests for normality.

- **$H_0$:** The data comes from a normal distribution.
- **$H_1$:** The data does NOT come from a normal distribution.
- If $p < 0.05$, we reject $H_0$ -- evidence against normality.

**Important caveat:** With large samples, even tiny deviations from normality become "significant." Always pair statistical tests with visual checks.

In [ ]:
print(f"{'Column':<20s} {'W-statistic':>12s} {'p-value':>10s}  {'Conclusion'}")
print("-" * 65)

for col in numeric_cols:
    data = df[col].dropna()
    w_stat, p_val = stats.shapiro(data)
    conclusion = "Normal (fail to reject H0)" if p_val >= 0.05 else "NOT normal (reject H0)"
    print(f"{col:<20s} {w_stat:>12.4f} {p_val:>10.4f}  {conclusion}")

print("\nNote: 'bill_length_mm' and 'flipper_length_mm' are likely non-normal due to")
print("the bimodal distribution caused by mixing species. Within each species, they")
print("may be closer to normal. This is an important lesson: check normality per group!")

In [ ]:
# Shapiro-Wilk per species -- to demonstrate the point
print("Shapiro-Wilk for bill_length_mm BY SPECIES:")
print(f"{'Species':<12s} {'W':>8s} {'p-value':>10s}  Conclusion")
print("-" * 50)
for species in ['Adelie', 'Chinstrap', 'Gentoo']:
    data = df.loc[df['species'] == species, 'bill_length_mm'].dropna()
    w, p = stats.shapiro(data)
    conc = "Normal" if p >= 0.05 else "NOT normal"
    print(f"{species:<12s} {w:>8.4f} {p:>10.4f}  {conc}")

#### QQ-Plots for Normality

A QQ-plot (quantile-quantile plot) compares the quantiles of your data against the quantiles of a theoretical normal distribution. If the data is normal, points lie on a straight line.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    data = df[col].dropna().values
    # Theoretical quantiles
    sorted_data = np.sort(data)
    n = len(sorted_data)
    theoretical_q = stats.norm.ppf(np.arange(1, n + 1) / (n + 1))
    
    ax.scatter(theoretical_q, sorted_data, s=15, alpha=0.6, color='steelblue')
    
    # Reference line (fit to the middle 50% of data)
    q25_idx = n // 4
    q75_idx = 3 * n // 4
    slope = (sorted_data[q75_idx] - sorted_data[q25_idx]) / (theoretical_q[q75_idx] - theoretical_q[q25_idx])
    intercept = sorted_data[q25_idx] - slope * theoretical_q[q25_idx]
    x_line = np.array([theoretical_q[0], theoretical_q[-1]])
    ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=2)
    
    ax.set_xlabel('Theoretical Quantiles (Normal)', fontsize=10)
    ax.set_ylabel(f'Sample Quantiles ({col})', fontsize=10)
    ax.set_title(f'QQ-Plot: {col}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('QQ-Plots for Normality Assessment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Reading QQ-plots:")
print("- Points on the red line = data matches normal distribution.")
print("- S-shaped curve = heavy tails or light tails.")
print("- Staircase at extremes = data has discrete values or bounded range.")
print("- Clear departure in the middle = skewness or multimodality.")

#### Outlier Detection: IQR Method vs Z-Score Method

Two classical methods:

1. **IQR method:** A point is an outlier if it is below $Q_1 - 1.5 \cdot \text{IQR}$ or above $Q_3 + 1.5 \cdot \text{IQR}$, where $\text{IQR} = Q_3 - Q_1$. This is what box plots use.

2. **Z-score method:** A point is an outlier if $|z| > 3$, i.e., more than 3 standard deviations from the mean.

The IQR method is more robust to extreme outliers because the median and quartiles are not affected by them.

In [ ]:
def detect_outliers_iqr(series):
    """Detect outliers using the IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper), lower, upper

def detect_outliers_zscore(series, threshold=3):
    """Detect outliers using the z-score method."""
    z = np.abs(stats.zscore(series.dropna()))
    # Return a boolean mask aligned with the original index
    mask = pd.Series(False, index=series.index)
    mask.loc[series.dropna().index] = z > threshold
    return mask

print(f"{'Column':<20s} {'IQR outliers':>12s} {'Z-score outliers':>16s}")
print("-" * 52)

for col in numeric_cols:
    data = df[col].dropna()
    iqr_mask, lo, hi = detect_outliers_iqr(data)
    z_mask = detect_outliers_zscore(df[col])
    print(f"{col:<20s} {iqr_mask.sum():>12d} {z_mask.sum():>16d}")

print("\nNote: IQR and z-score methods often disagree. IQR is based on")
print("quartiles (robust), while z-score is based on mean/std (sensitive to outliers).")
print("In practice, investigate flagged points rather than blindly removing them.")

In [ ]:
# Visualize outliers for body_mass_g
col = 'body_mass_g'
data = df[col].dropna()

iqr_mask, lo, hi = detect_outliers_iqr(data)
z_mask = detect_outliers_zscore(df[col])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IQR method
ax = axes[0]
ax.hist(data, bins=25, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(lo, color='red', linestyle='--', linewidth=2, label=f'Lower bound ({lo:.0f})')
ax.axvline(hi, color='red', linestyle='--', linewidth=2, label=f'Upper bound ({hi:.0f})')
ax.scatter(data[iqr_mask], np.zeros(iqr_mask.sum()) + 1, color='red', s=60, zorder=5, label=f'Outliers ({iqr_mask.sum()})')
ax.set_title('IQR Method', fontsize=12, fontweight='bold')
ax.set_xlabel(col)
ax.legend(fontsize=9)

# Z-score method
ax = axes[1]
z_data = df[col].dropna()
z_lo = data.mean() - 3 * data.std()
z_hi = data.mean() + 3 * data.std()
ax.hist(data, bins=25, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(z_lo, color='orange', linestyle='--', linewidth=2, label=f'Mean - 3*std ({z_lo:.0f})')
ax.axvline(z_hi, color='orange', linestyle='--', linewidth=2, label=f'Mean + 3*std ({z_hi:.0f})')
z_outlier_data = data[z_mask.loc[data.index]]
ax.scatter(z_outlier_data, np.zeros(len(z_outlier_data)) + 1, color='orange', s=60, zorder=5,
           label=f'Outliers ({len(z_outlier_data)})')
ax.set_title('Z-Score Method (|z| > 3)', fontsize=12, fontweight='bold')
ax.set_xlabel(col)
ax.legend(fontsize=9)

plt.suptitle(f'Outlier Detection: {col}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Scatter Plots -- Relationships Between Features

In [ ]:
# Scatter plot: bill length vs bill depth, colored by species
fig, ax = plt.subplots(figsize=(9, 6))

colors = {'Adelie': '#ff7f0e', 'Chinstrap': '#2ca02c', 'Gentoo': '#1f77b4'}

for species, color in colors.items():
    mask = df['species'] == species
    ax.scatter(
        df.loc[mask, 'bill_length_mm'],
        df.loc[mask, 'bill_depth_mm'],
        c=color, label=species, alpha=0.6, s=40, edgecolors='white', linewidth=0.5
    )

ax.set_xlabel('Bill Length (mm)', fontsize=12)
ax.set_ylabel('Bill Depth (mm)', fontsize=12)
ax.set_title('Bill Length vs Bill Depth by Species', fontsize=13, fontweight='bold')
ax.legend(title='Species', fontsize=10)
plt.tight_layout()
plt.show()

**Key observation:** The three species form distinct clusters! Gentoo penguins have long bills but shallow depth. This is exactly what a supervised learning model would learn to exploit.

In [ ]:
# Scatter plot: flipper length vs body mass
fig, ax = plt.subplots(figsize=(9, 6))

for species, color in colors.items():
    mask = df['species'] == species
    ax.scatter(
        df.loc[mask, 'flipper_length_mm'],
        df.loc[mask, 'body_mass_g'],
        c=color, label=species, alpha=0.6, s=40, edgecolors='white', linewidth=0.5
    )

ax.set_xlabel('Flipper Length (mm)', fontsize=12)
ax.set_ylabel('Body Mass (g)', fontsize=12)
ax.set_title('Flipper Length vs Body Mass by Species', fontsize=13, fontweight='bold')
ax.legend(title='Species', fontsize=10)
plt.tight_layout()
plt.show()

#### Box Plots -- Comparing Distributions Across Species

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

species_list = ['Adelie', 'Chinstrap', 'Gentoo']

for ax, col in zip(axes, numeric_cols):
    data_per_species = [df.loc[df['species'] == sp, col].dropna().values for sp in species_list]
    bp = ax.boxplot(data_per_species, labels=species_list, patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#ff7f0e', '#2ca02c', '#1f77b4']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### Correlation Heatmap

In [ ]:
# Compute correlation matrix for numeric columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)

# Labels
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(numeric_cols, fontsize=10)

# Annotate each cell with the correlation value
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        text_color = 'white' if abs(corr.iloc[i, j]) > 0.6 else 'black'
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=11, color=text_color, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson Correlation')
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations from the heatmap:**
- Flipper length and body mass are strongly positively correlated (~0.87) -- bigger penguins have longer flippers.
- Bill depth and bill length have a negative correlation when we look at all species together -- this is **Simpson's paradox!** Within each species the correlation may be positive, but Gentoo (long bills, shallow depth) pulls the overall correlation negative.

This is a great example of why EDA matters: a single number (the correlation) can be misleading without visualization.

### EXERCISE: Your Turn

1. Create a histogram of `body_mass_g` **per species** (3 histograms overlaid or side by side).
2. Which species is the heaviest on average? Compute it with pandas.
3. How many penguins have a bill length above 50 mm? What percentage of the total is that?

In [ ]:
# EXERCISE: Solution

# 1. Histograms of body_mass_g per species (overlaid)
fig, ax = plt.subplots(figsize=(9, 5))
for species, color in colors.items():
    data = df.loc[df['species'] == species, 'body_mass_g'].dropna()
    ax.hist(data, bins=20, alpha=0.5, color=color, label=species, edgecolor='white')
ax.set_xlabel('Body Mass (g)')
ax.set_ylabel('Count')
ax.set_title('Body Mass Distribution by Species')
ax.legend()
plt.tight_layout()
plt.show()

# 2. Heaviest species on average
print("Mean body mass per species:")
print(df.groupby('species')['body_mass_g'].mean().round(0))

# 3. Penguins with bill length > 50 mm
long_bills = (df['bill_length_mm'] > 50).sum()
pct = long_bills / len(df) * 100
print(f"\nPenguins with bill length > 50 mm: {long_bills} ({pct:.1f}%)")

---
## Part 3: Simpson's Paradox

We hinted at Simpson's paradox above. Now let's see it in full force with a controlled example.

**Simpson's paradox:** A trend that appears in aggregated data **reverses** when the data is separated into groups.

### Example: A Medical Treatment

A hospital runs a study on a new treatment. They collect recovery rates. Looking at the overall numbers, the treatment group has a *lower* recovery rate. But when they split by severity of illness, the treatment is *better* in every subgroup.

How is this possible? The treatment group had more severe cases (a confounding variable).

In [ ]:
np.random.seed(42)

# Build a dataset that exhibits Simpson's paradox
# Scenario: Treatment vs Control, Mild vs Severe illness

records = []

# Group 1: Mild illness, Control -- many patients, high recovery
n = 200
records.extend([{'group': 'Control', 'severity': 'Mild',
                 'recovered': np.random.rand() < 0.85} for _ in range(n)])

# Group 2: Mild illness, Treatment -- few patients, even higher recovery
n = 50
records.extend([{'group': 'Treatment', 'severity': 'Mild',
                 'recovered': np.random.rand() < 0.90} for _ in range(n)])

# Group 3: Severe illness, Control -- few patients, low recovery
n = 50
records.extend([{'group': 'Control', 'severity': 'Severe',
                 'recovered': np.random.rand() < 0.50} for _ in range(n)])

# Group 4: Severe illness, Treatment -- many patients, higher recovery
n = 200
records.extend([{'group': 'Treatment', 'severity': 'Severe',
                 'recovered': np.random.rand() < 0.60} for _ in range(n)])

simpson_df = pd.DataFrame(records)
simpson_df['recovered'] = simpson_df['recovered'].astype(int)

print(f"Dataset: {len(simpson_df)} patients")
print(f"Treatment: {(simpson_df['group'] == 'Treatment').sum()}, Control: {(simpson_df['group'] == 'Control').sum()}")

In [ ]:
# AGGREGATED view: Treatment looks WORSE
print("=" * 50)
print("AGGREGATED RECOVERY RATES (ignoring severity)")
print("=" * 50)
agg = simpson_df.groupby('group')['recovered'].agg(['sum', 'count', 'mean'])
agg.columns = ['recovered', 'total', 'recovery_rate']
print(agg)
print(f"\n--> Treatment recovery rate: {agg.loc['Treatment', 'recovery_rate']:.1%}")
print(f"--> Control recovery rate:   {agg.loc['Control', 'recovery_rate']:.1%}")
print("\n*** Treatment appears WORSE overall! ***")

In [ ]:
# DISAGGREGATED view: Treatment is BETTER in every subgroup
print("=" * 50)
print("DISAGGREGATED RECOVERY RATES (by severity)")
print("=" * 50)
disagg = simpson_df.groupby(['severity', 'group'])['recovered'].agg(['sum', 'count', 'mean'])
disagg.columns = ['recovered', 'total', 'recovery_rate']
print(disagg)

print("\n*** Treatment is BETTER in BOTH subgroups! ***")
print("\nThe paradox arises because Treatment patients were disproportionately")
print("assigned to the Severe group (200 out of 250). The Severe group has")
print("lower recovery rates regardless of treatment. This imbalance in group")
print("composition is the confounding variable.")

In [ ]:
# Visualize the paradox
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: aggregated
ax = axes[0]
agg_rates = simpson_df.groupby('group')['recovered'].mean()
bar_colors = ['#2ca02c', '#ff7f0e']
bars = ax.bar(agg_rates.index, agg_rates.values, color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, agg_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.1%}',
            ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Recovery Rate', fontsize=12)
ax.set_title('Aggregated: Treatment Looks Worse!', fontsize=13, fontweight='bold', color='red')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

# Right: disaggregated
ax = axes[1]
width = 0.35
x_pos = np.arange(2)
severities = ['Mild', 'Severe']

control_rates = [disagg.loc[(sev, 'Control'), 'recovery_rate'] for sev in severities]
treatment_rates = [disagg.loc[(sev, 'Treatment'), 'recovery_rate'] for sev in severities]

bars1 = ax.bar(x_pos - width/2, control_rates, width, label='Control', color='#2ca02c', edgecolor='white')
bars2 = ax.bar(x_pos + width/2, treatment_rates, width, label='Treatment', color='#ff7f0e', edgecolor='white')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.1%}', ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(severities, fontsize=12)
ax.set_ylabel('Recovery Rate', fontsize=12)
ax.set_title('By Severity: Treatment is Better in Both!', fontsize=13, fontweight='bold', color='green')
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

plt.suptitle("Simpson's Paradox: Aggregated Statistics Can Lie", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("Teaching point: ALWAYS segment your EDA. Aggregated statistics can reverse")
print("when you condition on a confounding variable. Before drawing conclusions,")
print("ask yourself: 'Is there a lurking variable that could change the story?'")

---
## Part 4: Correlation Is Not Causation (With Teeth)

Everyone has heard "correlation is not causation." Most people nod and then ignore it. Let's make it stick with three demonstrations.

### 4.1: Spurious Correlation -- Trending Time Series

Two completely independent time series can show a high Pearson correlation simply because they both trend upward. The correlation captures the shared trend, not any causal link.

In [ ]:
np.random.seed(42)

# Two INDEPENDENT trending time series
n = 100
t = np.arange(n)

# Series A: linear trend + noise (e.g., "ice cream sales")
series_a = 50 + 2 * t + np.random.normal(0, 15, n)

# Series B: completely independent linear trend + noise (e.g., "shark attacks")
series_b = 10 + 1.5 * t + np.random.normal(0, 12, n)

# Pearson correlation
r, p = stats.pearsonr(series_a, series_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series plot
ax = axes[0]
ax.plot(t, series_a, 'b-', alpha=0.7, label='Series A (e.g., ice cream sales)')
ax.plot(t, series_b, 'r-', alpha=0.7, label='Series B (e.g., shark attacks)')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('Two Independent Trending Series', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Scatter plot
ax = axes[1]
ax.scatter(series_a, series_b, alpha=0.5, s=20, color='steelblue')
# Regression line
m, b = np.polyfit(series_a, series_b, 1)
x_line = np.linspace(series_a.min(), series_a.max(), 100)
ax.plot(x_line, m * x_line + b, 'r-', linewidth=2)
ax.set_xlabel('Series A')
ax.set_ylabel('Series B')
ax.set_title(f'Scatter: r = {r:.3f} (p = {p:.1e})', fontsize=12, fontweight='bold')

plt.suptitle('Spurious Correlation from Shared Trends', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Pearson r = {r:.3f}, p-value = {p:.1e}")
print("\nThe correlation is high and 'statistically significant' -- but it is meaningless!")
print("Both series trend upward over time. The correlation captures the shared trend,")
print("not any causal relationship. Ice cream sales do not cause shark attacks.")
print("\nFix: detrend the series (subtract the linear trend), then check correlation.")

# Detrend and show the true correlation
detrend_a = series_a - np.polyval(np.polyfit(t, series_a, 1), t)
detrend_b = series_b - np.polyval(np.polyfit(t, series_b, 1), t)
r_detrend, p_detrend = stats.pearsonr(detrend_a, detrend_b)
print(f"\nAfter detrending: r = {r_detrend:.3f}, p = {p_detrend:.3f}")
print("Now the correlation is near zero -- as expected for independent series.")

### 4.2: Confounders

A **confounder** is a variable that influences both the apparent cause and the effect, creating a spurious association.

Example: "People who drink more coffee earn higher salaries." But age is a confounder -- older people drink more coffee AND earn more.

In [ ]:
np.random.seed(42)
n = 300

# Age is the confounder (range: 22 to 60)
age = np.random.uniform(22, 60, n)

# Coffee consumption depends on age (older -> more coffee, with noise)
coffee_cups = 1.0 + 0.05 * age + np.random.normal(0, 0.8, n)
coffee_cups = np.clip(coffee_cups, 0, 8)

# Salary depends on age (older -> higher salary, with noise), NOT on coffee
salary_k = 25 + 1.2 * age + np.random.normal(0, 10, n)

# Create DataFrame
conf_df = pd.DataFrame({'age': age, 'coffee_cups_per_day': coffee_cups, 'salary_k': salary_k})

# Naive correlation: coffee vs salary
r_naive, p_naive = stats.pearsonr(conf_df['coffee_cups_per_day'], conf_df['salary_k'])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Naive: coffee vs salary
ax = axes[0]
ax.scatter(conf_df['coffee_cups_per_day'], conf_df['salary_k'], alpha=0.4, s=20, color='steelblue')
m, b = np.polyfit(conf_df['coffee_cups_per_day'], conf_df['salary_k'], 1)
x_line = np.linspace(0, 8, 100)
ax.plot(x_line, m * x_line + b, 'r-', linewidth=2)
ax.set_xlabel('Coffee (cups/day)')
ax.set_ylabel('Salary ($k)')
ax.set_title(f'Naive: r = {r_naive:.3f}\nCoffee causes wealth?', fontsize=11, fontweight='bold')

# 2. The confounder revealed: age vs coffee, age vs salary
ax = axes[1]
sc = ax.scatter(conf_df['age'], conf_df['salary_k'], c=conf_df['coffee_cups_per_day'],
                cmap='YlOrRd', alpha=0.6, s=25)
plt.colorbar(sc, ax=ax, label='Coffee cups/day')
ax.set_xlabel('Age')
ax.set_ylabel('Salary ($k)')
ax.set_title('Confounder: Age drives both!', fontsize=11, fontweight='bold')

# 3. Controlling for age: residualize both variables
coffee_resid = conf_df['coffee_cups_per_day'] - np.polyval(np.polyfit(age, coffee_cups, 1), age)
salary_resid = conf_df['salary_k'] - np.polyval(np.polyfit(age, salary_k, 1), age)
r_controlled, p_controlled = stats.pearsonr(coffee_resid, salary_resid)

ax = axes[2]
ax.scatter(coffee_resid, salary_resid, alpha=0.4, s=20, color='steelblue')
m2, b2 = np.polyfit(coffee_resid, salary_resid, 1)
x_line2 = np.linspace(coffee_resid.min(), coffee_resid.max(), 100)
ax.plot(x_line2, m2 * x_line2 + b2, 'r-', linewidth=2)
ax.set_xlabel('Coffee (age-adjusted residual)')
ax.set_ylabel('Salary (age-adjusted residual)')
ax.set_title(f'Controlled for age: r = {r_controlled:.3f}\nNo real effect!', fontsize=11, fontweight='bold')

plt.suptitle('Confounding Variable: Age Drives Both Coffee and Salary', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print(f"Naive correlation (coffee vs salary):    r = {r_naive:.3f}")
print(f"After controlling for age (residuals):   r = {r_controlled:.3f}")
print("\nThe apparent effect of coffee on salary vanishes once we account for age.")
print("Lesson: Before claiming X causes Y, ask what variable Z could explain both.")

### 4.3: Anscombe's Quartet

Four datasets with (nearly) identical summary statistics -- same mean, variance, correlation, and linear regression line -- but wildly different distributions. This is the classic argument for **always plotting your data**.

In [ ]:
# Anscombe's quartet: exact values
anscombe_x = {
    'I':   [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'II':  [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'III': [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'IV':  [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
}

anscombe_y = {
    'I':   [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    'II':  [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    'III': [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    'IV':  [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
}

# Show that all four have the same statistics
print(f"{'Dataset':<8s} {'x_mean':>7s} {'y_mean':>7s} {'x_std':>7s} {'y_std':>7s} {'r':>7s} {'slope':>7s} {'intercept':>9s}")
print("-" * 62)
for key in ['I', 'II', 'III', 'IV']:
    x = np.array(anscombe_x[key])
    y = np.array(anscombe_y[key])
    r = np.corrcoef(x, y)[0, 1]
    m, b = np.polyfit(x, y, 1)
    print(f"{key:<8s} {x.mean():>7.2f} {y.mean():>7.2f} {x.std():>7.2f} {y.std():>7.2f} {r:>7.3f} {m:>7.3f} {b:>9.3f}")

print("\nAll four datasets have nearly identical statistics!")
print("But look at the scatter plots...")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (key, ax) in enumerate(zip(['I', 'II', 'III', 'IV'], axes)):
    x = np.array(anscombe_x[key])
    y = np.array(anscombe_y[key])
    
    ax.scatter(x, y, s=60, color='steelblue', edgecolors='white', linewidth=0.5, zorder=5)
    
    # Same regression line for all
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(2, 20, 100)
    ax.plot(x_line, m * x_line + b, 'r-', linewidth=1.5, alpha=0.7)
    
    r = np.corrcoef(x, y)[0, 1]
    ax.set_title(f'Dataset {key} (r = {r:.3f})', fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)
    ax.grid(True, alpha=0.3)

plt.suptitle("Anscombe's Quartet: Same Statistics, Different Stories", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Dataset I:   Linear relationship (correlation is appropriate).")
print("Dataset II:  Curved relationship (linear model is wrong).")
print("Dataset III: Perfect linear except for one outlier that skews everything.")
print("Dataset IV:  No relationship -- a single outlier creates the illusion of correlation.")
print("\nLesson: NEVER trust summary statistics alone. Always. Plot. Your. Data.")

---
## Part 5: NumPy Warm-up

### Why Vectorization Matters

Python `for` loops are slow because Python is an interpreted language. NumPy operations run in compiled C code under the hood -- they are **orders of magnitude faster**.

This matters for ML: training a neural network involves millions of arithmetic operations per second.

In [ ]:
# Task: compute the square of each element in an array of 1 million numbers

size = 1_000_000
data = np.random.randn(size)

In [ ]:
# Method 1: Python for-loop
def square_loop(arr):
    result = np.empty(len(arr))
    for i in range(len(arr)):
        result[i] = arr[i] ** 2
    return result

# Time it (1 run since it is slow)
%timeit -n 1 -r 3 square_loop(data)

In [ ]:
# Method 2: NumPy vectorized
def square_vectorized(arr):
    return arr ** 2

%timeit square_vectorized(data)

In [ ]:
# Quick comparison (approximate, actual numbers come from %timeit above)
import time

start = time.time()
_ = square_loop(data)
loop_time = time.time() - start

start = time.time()
_ = square_vectorized(data)
vec_time = time.time() - start

print(f"Loop:       {loop_time:.4f} seconds")
print(f"Vectorized: {vec_time:.6f} seconds")
print(f"Speedup:    {loop_time / vec_time:.0f}x")

### Broadcasting

Broadcasting lets NumPy perform operations on arrays of different shapes without copying data.

**Rule:** Dimensions are compared from the right. They must be equal, or one of them must be 1.

In [ ]:
# Example: add a row vector to every row of a matrix
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])

row_vector = np.array([10, 20, 30])

print(f"Matrix shape:     {matrix.shape}")      # (3, 3)
print(f"Row vector shape: {row_vector.shape}")   # (3,)

result = matrix + row_vector  # Broadcasting! row_vector is added to each row
print(f"\nResult:\n{result}")

In [ ]:
# Another example: multiply each column by a different scalar
col_vector = np.array([[2],
                        [3],
                        [4]])

print(f"Matrix shape:     {matrix.shape}")       # (3, 3)
print(f"Col vector shape: {col_vector.shape}")    # (3, 1)

result = matrix * col_vector  # Each row is multiplied by a different scalar
print(f"\nResult:\n{result}")

### EXERCISE: Normalize Columns Using Broadcasting

Given a matrix, subtract the column mean from each column (so each column has mean ~0).

Do this **without a for-loop** -- use broadcasting.

In [ ]:
# EXERCISE: Solution

# Create a sample matrix
X = np.random.randint(1, 100, size=(5, 4)).astype(float)
print("Original matrix:")
print(X)
print(f"\nColumn means: {X.mean(axis=0)}")

# Normalize: subtract column means (broadcasting!)
X_centered = X - X.mean(axis=0)  # shape (5,4) - shape (4,) => broadcasting works

print(f"\nCentered matrix:")
print(X_centered.round(1))
print(f"\nColumn means after centering: {X_centered.mean(axis=0).round(10)}")
# Should be ~0 (within floating point precision)

---
## Part 6: Additional Exercises

These exercises are more challenging. They integrate multiple concepts from today's session.

### Exercise A: Normality Assessment

Given a dataset with several features, identify which are approximately normal and which are not. Use both visual (QQ-plots) and statistical (Shapiro-Wilk) methods.

In [ ]:
# Generate a dataset with features of different distributions
np.random.seed(123)
n_samples = 200

exercise_df = pd.DataFrame({
    'feature_A': np.random.normal(50, 10, n_samples),          # Normal
    'feature_B': np.random.exponential(5, n_samples),          # Exponential (right-skewed)
    'feature_C': np.random.normal(0, 1, n_samples),            # Normal
    'feature_D': np.random.uniform(0, 100, n_samples),         # Uniform (not normal)
    'feature_E': np.random.lognormal(2, 0.5, n_samples),       # Log-normal (right-skewed)
})

print("Dataset with 5 features. YOUR TASK:")
print("1. For each feature, create a QQ-plot and run the Shapiro-Wilk test.")
print("2. Classify each as approximately normal or not.")
print("3. For non-normal features, describe what kind of departure you see.")
print("\nSample data:")
exercise_df.head()

In [ ]:
# YOUR CODE HERE


### Exercise B: Outlier Impact on Mean vs Median

Demonstrate how outliers affect the mean but not the median. Given a dataset, detect outliers, then compare the mean and median before and after removing them.

In [ ]:
np.random.seed(99)

# Create data with some extreme outliers
normal_data = np.random.normal(100, 15, 200)
# Inject 5 extreme outliers
outliers = np.array([250, 280, 300, 10, -20])
all_data = np.concatenate([normal_data, outliers])
np.random.shuffle(all_data)

print("TASK:")
print("1. Compute mean and median of the data.")
print("2. Detect outliers using the IQR method.")
print("3. Remove outliers and recompute mean and median.")
print("4. How much did each change? Which is more robust?")
print(f"\nData: {len(all_data)} observations, range [{all_data.min():.1f}, {all_data.max():.1f}]")

In [ ]:
# YOUR CODE HERE


### Exercise C: Find Simpson's Paradox

You are given data on student performance in two teaching methods (Online vs In-Person) across two departments (Engineering and Arts). Your task: compute the overall pass rate by method, then the pass rate by department. Find the reversal.

In [ ]:
np.random.seed(77)

# Create a Simpson's paradox dataset for students
student_records = []

# Engineering + Online: small group, high pass rate
for _ in range(30):
    student_records.append({'department': 'Engineering', 'method': 'Online',
                           'passed': int(np.random.rand() < 0.75)})

# Engineering + In-Person: large group, even higher pass rate
for _ in range(180):
    student_records.append({'department': 'Engineering', 'method': 'In-Person',
                           'passed': int(np.random.rand() < 0.80)})

# Arts + Online: large group, low pass rate
for _ in range(170):
    student_records.append({'department': 'Arts', 'method': 'Online',
                           'passed': int(np.random.rand() < 0.40)})

# Arts + In-Person: small group, slightly higher pass rate
for _ in range(40):
    student_records.append({'department': 'Arts', 'method': 'In-Person',
                           'passed': int(np.random.rand() < 0.45)})

student_df = pd.DataFrame(student_records)

print("TASK:")
print("1. Compute the overall pass rate by method (Online vs In-Person).")
print("2. Compute the pass rate by method WITHIN each department.")
print("3. Do you see a reversal? Explain why it happens.")
print(f"\nDataset: {len(student_df)} students")
print(student_df.groupby(['department', 'method']).size().unstack(fill_value=0))

In [ ]:
# YOUR CODE HERE


---
## Wrap-up

### Today's Key Takeaways

1. **ML learns patterns from data.** Three types: supervised (labels), unsupervised (structure), reinforcement learning (rewards).
2. **Always start with EDA.** Look at your data with `.head()`, `.info()`, `.describe()`, and visualizations before building any model. Go beyond eyeballing: use skewness, kurtosis, Shapiro-Wilk tests, and QQ-plots.
3. **Bias vs variance.** Too simple = underfitting. Too complex = overfitting. We proved the decomposition $E[(y-\hat{f})^2] = \text{Bias}^2 + \text{Var} + \sigma^2$ and verified it empirically.
4. **Aggregated statistics lie.** Simpson's paradox, spurious correlations, and confounders are real. Always segment your EDA and think about lurking variables.
5. **Always plot your data.** Anscombe's quartet is the definitive argument.
6. **Vectorize with NumPy.** Never loop over array elements in Python. Broadcasting is your friend.

### Homework

Pick a dataset and perform a full EDA. Details in `homework.md`. **Due before Session 3 (Mon Mar 9).**

Commit your work to `homework/session-01/homework.ipynb` in your student repo.

### Before Next Session (Wed Mar 4)

Review the required support material:
- **3Blue1Brown: Essence of Linear Algebra, chapters 1--3** (~40 min) -- vectors, linear combinations, span.

**Next session:** Linear regression, loss functions, and gradient descent. We go from exploring data to making predictions.

In [ ]:
# Git commit reminder!
# In your terminal:
#   git add .
#   git commit -m "Session 1: ML landscape and EDA"
#   git push